<a href="https://colab.research.google.com/github/BraedynL0530/CanopyWatch/blob/master/deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#network stuff here when i leave collab

In [ ]:
import io
import requests
import rasterio
import numpy as np

#DO NOT RUN LOCALLY MY PC WILL EXPLODE!!!!!!!!!!
fastapi_url = "http://127.0.0.1:8000/api/get-satellite-patch" #wont work in google collab(unless i tunnell) this is for deploy/temp link
print("Fetching data from server...")
response = requests.get(fastapi_url)

if response.status_code == 204:
    print("No large NVDI change")
    exit()
elif response.status_code != 200:
    print(f"Error fetching: {response.status_code}")
    exit()

tiff_bytes = response.content

with rasterio.open(io.BytesIO(tiff_bytes)) as src:
    print("--- Raster Metadata ---")
    print(f"Width x height: {src.width}x{src.height}")
    print(f"Band Count: {src.count}")
    print(f"Coordinate System: {src.crs}")
    print(f"Geotransform Matrix:\n{src.transform}")
    print("-----------------------")

    #Shape: (4, 512, 512)  (Bands, Height, Width)
    full_stack = src.read()

    #slice the first 3 bands out for visual evidence / frontend
    rgb_channels = full_stack[0:3, :, :]  # Shape: (3, 512, 512)


    nir_channel = full_stack[3, :, :]     # Shape: (512, 512)

print("!!unpacked channels!!")

In [ ]:
#model
import torch
import torch.nn as nn

class forestClassifier(nn.Module):
    def __init__(self):
        super(forestClassifier, self).__init__()
        self.encoder = nn.Conv2d(4, 64, kernel_size=3,  padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneckConv = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.upsample = nn.ConvTranspose2d(128,64,kernel_size=2,stride=2)

        self.finalConv = nn.Conv2d(64,1,kernel_size=1)

        self.relu = nn.ReLU()
    def forward(self, x):
        x1 = self.relu(self.encoder(x))
        x2 = self.pool(x1)
        b=self.relu(self.bottleneckConv(x2))
        u=self.relu(self.upsample(b))
        out = self.finalConv(u)
        return out



In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from datasets import load_dataset
import torch.optim as optim
import rasterio
from rasterio.enums import Resampling

class DeforestationDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # We can convert them directly to numpy arrays.
        img_np = np.array(item['image'], dtype=np.float32)
        mask_np = np.array(item['label'], dtype=np.float32)

        if img_np.max() > 255.0:
            img_np = img_np / 10000.0
        else:
            img_np = img_np / 255.0

        if len(img_np.shape) == 3 and img_np.shape[-1] in [3, 4]:
            img_np = np.transpose(img_np, (2, 0, 1))
        elif len(img_np.shape) == 2:
            img_np = np.expand_dims(img_np, axis=0)

        # Pad to 4 channels if necessary
        if img_np.shape[0] == 3:
            nir_pad = np.zeros((1, img_np.shape[1], img_np.shape[2]), dtype=np.float32)
            img_np = np.concatenate((img_np, nir_pad), axis=0)
        elif img_np.shape[0] > 4:
            img_np = img_np[:4, :, :]

        if len(mask_np.shape) == 2:
            mask_np = np.expand_dims(mask_np, axis=0)
        elif len(mask_np.shape) == 3:
            if mask_np.shape[-1] in [1, 3]:
                mask_np = np.transpose(mask_np, (2, 0, 1))
            mask_np = mask_np[0:1, :, :]

        mask_np = (mask_np > 0.0).astype(np.float32)

        return torch.tensor(img_np), torch.tensor(mask_np)

def train():
    print("Downloading/Loading Dataset from Hugging Face...")
    dataset = load_dataset("IsuruDiIshan/amazon-sentinel2-forest-segmentation")

    train_data = dataset['train']
    val_data = dataset['val']

    print(f"Loaded {len(train_data)} training and {len(val_data)} validation samples.")

    train_dataset = DeforestationDataset(train_data)
    val_dataset = DeforestationDataset(val_data)

    if len(train_dataset) == 0:
        print("Dataset is empty. Check split names.")
        return

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = forestClassifier().to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

    epochs = 15 #bigger legues
    print(f"Starting training on {device}...")

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch_idx, (images, masks) in enumerate(train_loader):
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            predictions = model(images)
            loss = criterion(predictions, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                predictions = model(images)
                loss = criterion(predictions, masks)
                val_loss += loss.item()


        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        scheduler.step(avg_val)

        print(f"--- Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} ---")

    print("Saving weights...")
    torch.save(model.state_dict(), 'canopywatch_v1.pth')

if __name__ == "__main__":
    train()

In [1]:
#infrence gemni  made for me since i forgot to put it in model:model.eval() this is more of a testing script tho
import matplotlib.pyplot as plt

def run_inference(weights_path='canopywatch_v1.pth'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 1. Load Model
    model = forestClassifier().to(device)
    try:
        model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
        print("Weights loaded successfully!")
    except FileNotFoundError:
        print(f"Error: Could not find '{weights_path}'. Make sure you've trained the model first!")
        return

    model.eval() # Essential for inference

    # 2. Fetch Validation Data from Hugging Face
    print("Downloading/Loading Validation Data from Hugging Face...")
    dataset = load_dataset("IsuruDiIshan/amazon-sentinel2-forest-segmentation", split='val')

    # 3. Pick a random sample
    idx = np.random.randint(0, len(dataset))
    item = dataset[idx]

    # Extract and format the image exactly like we did in training
    img_np = np.array(item['image'], dtype=np.float32)
    if img_np.max() > 255.0:
        img_np = img_np / 10000.0
    else:
        img_np = img_np / 255.0

    # FIX: Only transpose if the array is (H, W, C)
    if len(img_np.shape) == 3 and img_np.shape[-1] in [3, 4]:
        img_np = np.transpose(img_np, (2, 0, 1))

    if img_np.shape[0] == 3:
        nir_pad = np.zeros((1, img_np.shape[1], img_np.shape[2]), dtype=np.float32)
        img_np = np.concatenate((img_np, nir_pad), axis=0)
    elif img_np.shape[0] > 4:
        img_np = img_np[:4, :, :]

    mask_np = np.array(item['label'], dtype=np.float32)

    # Convert to tensor and add batch dimension -> [1, 4, 512, 512]
    img_tensor = torch.tensor(img_np).unsqueeze(0).to(device)

    # 4. Run through the model
    print(f"Running inference on validation image #{idx}...")
    with torch.no_grad():
        logits = model(img_tensor)
        # Apply sigmoid to convert raw logits to probabilities (0.0 to 1.0)
        prediction = torch.sigmoid(logits).squeeze().cpu().numpy()

    # 5. Visualization
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    # Extract RGB bands for visualization and artificially brighten them
    # (Raw satellite imagery often looks pitch black without scaling)
    rgb_img = img_np[:3, :, :].transpose(1, 2, 0)
    rgb_display = np.clip(rgb_img * 2.5, 0, 1) # 2.5x brightness multiplier

    axes[0].imshow(rgb_display)
    axes[0].set_title(f"Input RGB (Val Image #{idx})", fontsize=14)
    axes[0].axis('off')

    # Ground Truth Mask
    axes[1].imshow(mask_np, cmap='gray')
    axes[1].set_title("Ground Truth (Actual Forest)", fontsize=14)
    axes[1].axis('off')

    # Model Prediction (Heatmap)
    im3 = axes[2].imshow(prediction, cmap='viridis', vmin=0, vmax=1)
    axes[2].set_title("U-Net Prediction (Confidence Score)", fontsize=14)
    axes[2].axis('off')

    # Add a colorbar to the prediction to show confidence levels
    cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    cbar.set_label('Forest Probability', rotation=270, labelpad=15)

    # New: Overlay Panel
    axes[3].imshow(rgb_display)
    # Mask out low-confidence predictions so we only overlay the confident forest pixels in green
    overlay_mask = np.ma.masked_where(prediction < 0.5, prediction)
    axes[3].imshow(overlay_mask, cmap='Greens', alpha=0.5, vmin=0, vmax=1)
    axes[3].set_title("AI Overlay (Green = Forest)", fontsize=14)
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_inference('canopywatch_v1.pth')

NameError: name 'torch' is not defined